# Live portfolio tearsheet

Reads the `performance.live_log` ledger and displays portfolio-first metrics,
then per-strategy sleeves (starting with `s1_equities`).

Requires equity rows from the S1 paper runner and/or
`python -m performance.eod_snapshot` (or `09_performance/eod_snapshot.py`).


## 0. Imports & Config


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from performance.live_log import (
    STRATEGY_S1_EQUITIES,
    get_meta,
    load_equity_daily,
    load_fills,
    load_signals,
    portfolio_equity_series,
    strategy_equity_series,
)
from performance.metrics import (
    equity_to_returns,
    live_portfolio_metrics,
    live_strategy_metrics,
    plot_rolling_metrics,
    summary_metrics_table,
)

ROLLING_LOOKBACK = 21
FREQ = "d"
YEAR_FREQ = "252d"

meta = get_meta()
sleeve_ids = list((meta.get("sleeve_weights") or {STRATEGY_S1_EQUITIES: 1.0}).keys())
print("portfolio_go_live=", meta.get("portfolio_go_live"))
print("sleeve_weights=", meta.get("sleeve_weights"))
print("sleeves=", sleeve_ids)


## 1. Data Loading


In [ ]:
equity_daily = load_equity_daily()
port_equity = portfolio_equity_series()

if equity_daily.empty or port_equity.empty:
    raise SystemExit(
        "live_log equity_daily is empty. Run the S1 paper runner and/or "
        "`python -m performance.eod_snapshot` before this tearsheet."
    )

display(equity_daily.tail())
port_equity.tail()


## 2. Portfolio Metrics


In [ ]:
port = live_portfolio_metrics(freq=FREQ, year_freq=YEAR_FREQ)
if port["summary"] is None:
    raise SystemExit(
        "Need at least two portfolio equity points to compute returns/metrics."
    )

display(port["summary"].to_frame("portfolio"))

fig, ax = plt.subplots(figsize=(11, 3.5), constrained_layout=True)
ax.plot(port["equity"].index, port["equity"].values, color="#1f4e79", lw=1.6)
ax.set_title("Live portfolio equity", loc="left")
ax.set_ylabel("Equity")
ax.grid(True, axis="y", alpha=0.4)
plt.show()

plot_rolling_metrics(
    port["returns"],
    ROLLING_LOOKBACK,
    freq=FREQ,
    year_freq=YEAR_FREQ,
    title="Live portfolio rolling metrics",
)


## 3. Per-Strategy Metrics


In [ ]:
summaries = {}
for sid in sleeve_ids:
    out = live_strategy_metrics(sid, freq=FREQ, year_freq=YEAR_FREQ)
    print(f"\n=== {sid} ===")
    if out["summary"] is None:
        print("Insufficient sleeve equity history.")
        continue
    summaries[sid] = out["summary"]
    display(out["summary"].to_frame(sid))

    fig, ax = plt.subplots(figsize=(11, 3.0), constrained_layout=True)
    ax.plot(out["equity"].index, out["equity"].values, color="#1f4e79", lw=1.6)
    ax.set_title(f"Sleeve equity: {sid}", loc="left")
    ax.set_ylabel("Equity")
    ax.grid(True, axis="y", alpha=0.4)
    plt.show()

    plot_rolling_metrics(
        out["returns"],
        ROLLING_LOOKBACK,
        freq=FREQ,
        year_freq=YEAR_FREQ,
        title=f"Live strategy rolling metrics: {sid}",
    )

if summaries:
    display(pd.DataFrame(summaries))


## 4. Recent Signals & Fills (Audit)


In [ ]:
signals = load_signals()
fills = load_fills()

print(f"signals rows={len(signals)}  fills rows={len(fills)}")
if not signals.empty:
    display(signals.sort_values(["decision_date", "ticker"]).tail(20))
if not fills.empty:
    display(fills.sort_values("ts").tail(20))
